In [0]:
table_filters = {
    "AdMessages"             : None,
    "AdminChatUsers"          : None,
    "AdminMasterUser"        : None,
    "AwardCoins"             : None,
    "ChatBlockedWords"       : None,
    "ChatHistory"            : "ClassID IS NOT NULL",
    "ClassRoomSetup"         : None,
    "ClassStudent"           : None,
    "Configtable"           : None,
    "FavoriteActivity"       : None,
    "GameFonts"              : None,
    "GameMaster"             : None,
    "GameQuestions"          : None,
    "GameSessionlines"       : None,
    "GameSessions"           : None,
    "MediaLibray"            : None,
    "MessageList"            : None,
    "PopupMessages"          : None,
    "ppsvlogmessages"        : None,
    "PurchasedGame"          : None,
    "PurchasedProfileAvatar" : None,
    "ReportChatMessage"      : None,
    "StudentAccessCodeLogs"  : None,
    "StudentActionHistory"   : None,
    "StudentAdAction"        : None,
    "StudentProfile"         : ["Profile_id IS NOT NULL", "StudentProfileID IS NOT NULL"]
}
print("all tables excuted")

In [0]:
from pyspark.sql.functions import expr, current_timestamp

for table, condition in table_filters.items():
    print(f"Processing table: {table}")
    df = spark.table(f"healthcareNew.bronze.{table}")

     # Case 1: No filter
    if condition is None:
        pass  # do nothing
    
    # Case 2: Multiple filters (list)
    elif isinstance(condition, list):
        for cond in condition:
            df = df.filter(expr(cond))
    
    # Case 3: Single filter (string)
    else:
        df = df.filter(expr(condition))
    
    # Remove duplicates (optional)
    df = df.dropDuplicates()
    
    # Add audit column
    df = df.withColumn("processed_at", current_timestamp())

    # Save to Silver
    df.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(f"healthcarenew.silver.{table}")
    
    print(f"{table} loaded to silver")
    